In [ ]:
# Cell 1: Setup
import polars as pl
import spatial_split as ss
REGION = 'HIM'
STRESS_YEARS = [2003, 2001]   # weakest years since 2000 for HIM (78% / 82% of rows pass the quality gate); one is added to the search years
N_SPREAD = 6                  # evenly spread search years

In [ ]:
# Cell 2: Which years are usable for this region? (drops data-sparse years at the start AND the end)
usable, table = ss.usable_years(REGION, frac=0.85, min_cls_frac=0.75)
print(table.to_pandas().to_string(index=False))
print('\nusable years:', usable)

In [ ]:
# Cell 3: Search years = evenly spread usable years + one weak "stress" year. Check the lists, then save.
search = ss.spread(usable, N_SPREAD)
stress = next((y for y in STRESS_YEARS if y in usable and y not in search), None)
if stress: search = sorted(search + [stress])
print('search years:', search, '| stress year:', stress)
ss.save_study_years(REGION, usable, search, table)

In [ ]:
# Cell 4: Build the polygon-level split on the usable years (gated rows). ~minutes. Old lookups are not reused (year set is part of the file name).
ss.build_block_splits(REGION, force=True)

In [ ]:
# Cell 5: Check the split per year. Look for years where val/test miss a class, or are tiny.
s = ss.split_summary(REGION)
print(s.to_pandas().to_string(index=False))

In [ ]:
# Cell 6: Polygon sanity check (same polygon -> one class; polygons stay whole in one split)
ss.polygon_check(REGION, years=ss.study_years(REGION)[:3])